this dataset is really small but i wanted to practice before starting on larger datasets cuz i realised i wasnt ready.

In [181]:
import pandas as pd

df_raw = pd.read_csv('Dataset.csv')
df= df_raw.copy()
display(df)

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]
5,6,2[4],10[9],"$305,158,363","$388,978,496",Madonna,The MDNA Tour,2012,88,"$3,467,709",[9]
6,7,2[10],NaN,"$280,000,000","$381,932,682",Celine Dion,Taking Chances World Tour,2008–2009,131,"$2,137,405",[11]
7,7,NaN,NaN,"$257,600,000","$257,600,000",Pink,Summer Carnival †,2023–2024,41,"$6,282,927",[12]
8,9,NaN,NaN,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215",[13]
9,10,NaN,NaN,"$250,400,000","$309,141,878",Taylor Swift,The 1989 World Tour,2015,85,"$2,945,882",[14]


### Cleaning Headers

In [182]:
# Convert column names to lowercase, replace spaces and non-breaking spaces with underscores, and remove punctuation
df.columns = [col.lower() for col in df.columns]
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('\xa0', '_').str.replace(r'([.,])', '', regex=True)
display(df.columns)

Index(['rank', 'peak', 'all_time_peak', 'actual_gross',
       'adjusted_gross_(in_2022_dollars)', 'artist', 'tour_title', 'year(s)',
       'shows', 'average_gross', 'ref'],
      dtype='object')

while cleaning the headers i learned that theres a character that is similar to " " but its not just a space. which is \xa0. we include that and replace it with "_"

### Dropping Unnecessary Columns

In [183]:
# Drop 'ref', 'peak', 'all_time_peak', 'rank' and 'average_gross' columns
df.drop(columns=['ref', 'peak', 'all_time_peak', 'rank', 'average_gross'], inplace=True)

### Cleaning and Extracting Years from 'year(s)'

year(s) is really messy right now might be because it was scraped from a website. we first extract the main years from the values. assign first one to start_year and second one end_year. if theres only one value, we assign that value to both of the new colunms which means the duration was within a year.

In [184]:
# Clean and extract years
df['year(s)'] = df['year(s)'].str.strip().str.replace(r'([.,])', '', regex=True)

df['start_year'] = df['year(s)'].str.extract(r'(\d{4})')
df['end_year'] = df['year(s)'].str.extract(r'\d{4}\D*(\d{4})')

df['end_year'] = df['end_year'].fillna(df['start_year'])

df['start_year'] = pd.to_numeric(df['start_year'], errors='coerce').astype('Int64')
df['end_year'] = pd.to_numeric(df['end_year'], errors='coerce').astype('Int64')

df['start_year'] = df['start_year'].fillna(df['end_year'])

df.drop(columns=['year(s)'], inplace=True)

### Cleaning and Renaming 'adjusted_gross_(in_2022_dollars)'

Remove non-digit characters from 'adjusted_gross_(in_2022_dollars)' and convert to numeric and then renaming it for clarity (optional)

In [185]:

df['adjusted_gross_(in_2022_dollars)'] = df['adjusted_gross_(in_2022_dollars)'].str.replace(r'[^\d]', '', regex=True)
df['adjusted_gross_(in_2022_dollars)'] = pd.to_numeric(df['adjusted_gross_(in_2022_dollars)'], errors='coerce')

df.rename(columns={'adjusted_gross_(in_2022_dollars)': 'adjusted_gross'}, inplace=True)

### Processing 'tour_title' and 'artist' Columns

Storing tour_title as a lis cuz we might need it later. useful information shouldnt be dropped recklessly. then dropping actual_gross and tour_title from df

In [186]:

tour_titles = df['tour_title'].to_list().copy()
df.drop(columns=['actual_gross', 'tour_title'], inplace=True)

# cleaning artist (optuional)
df['artist'] = [col.lower() for col in df['artist']]
df['artist'] = [col.strip() for col in df['artist']]


df = pd.get_dummies(df, columns=['artist'])

### Final DataFrame Information and Head

In [187]:
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   adjusted_gross       20 non-null     int64
 1   shows                20 non-null     int64
 2   start_year           20 non-null     Int64
 3   end_year             20 non-null     Int64
 4   artist_adele         20 non-null     bool 
 5   artist_beyoncé       20 non-null     bool 
 6   artist_celine dion   20 non-null     bool 
 7   artist_cher          20 non-null     bool 
 8   artist_katy perry    20 non-null     bool 
 9   artist_lady gaga     20 non-null     bool 
 10  artist_madonna       20 non-null     bool 
 11  artist_pink          20 non-null     bool 
 12  artist_taylor swift  20 non-null     bool 
dtypes: Int64(2), bool(9), int64(2)
memory usage: 992.0 bytes


,adjusted_gross,shows,start_year,end_year,artist_adele,artist_beyoncé,artist_celine dion,artist_cher,artist_katy perry,artist_lady gaga,artist_madonna,artist_pink,artist_taylor swift
0,780000000,56,2023,2024,False,False,False,False,False,False,False,False,True
1,579800000,56,2023,2023,False,True,False,False,False,False,False,False,False
2,560622615,85,2008,2009,False,False,False,False,False,False,True,False,False
3,454751555,156,2018,2019,False,False,False,False,False,False,False,True,False
4,402844849,53,2018,2018,False,False,False,False,False,False,False,False,True


In [188]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   adjusted_gross       20 non-null     int64
 1   shows                20 non-null     int64
 2   start_year           20 non-null     Int64
 3   end_year             20 non-null     Int64
 4   artist_adele         20 non-null     bool 
 5   artist_beyoncé       20 non-null     bool 
 6   artist_celine dion   20 non-null     bool 
 7   artist_cher          20 non-null     bool 
 8   artist_katy perry    20 non-null     bool 
 9   artist_lady gaga     20 non-null     bool 
 10  artist_madonna       20 non-null     bool 
 11  artist_pink          20 non-null     bool 
 12  artist_taylor swift  20 non-null     bool 
dtypes: Int64(2), bool(9), int64(2)
memory usage: 992.0 bytes


our dataset is ready for ML.

### Machine Learning

In [189]:
X = df.drop(columns=['adjusted_gross'])
y = df['adjusted_gross']

In [190]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

using simple linear model like linear regression because accuracy is not the goal here

In [191]:
from sklearn.linear_model import LinearRegression
model= LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [192]:
y_pred = model.predict(x_test)

###Evaluation

In [193]:
from sklearn.metrics import mean_absolute_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 301754045.37425613
R2: -1.2099115711552058


the model sucks becuase we have very tiny data so model is basically memorising the data instead of finding patterns. however the goal here was to get comfortable in data cleaning which was achieved. data cleaning taught a lot of stuff today. will try to clean a larger data tommorro since this was succesful unlike last time where i had to to AI's help though i coded myself.